# garimpo-uncertainty on Google Colab

This notebook reproduces the experiment on Google Colab, in the order of a training notebook:
setup, data, run configs, tests, pilot, training of the four models, evaluation, and the release
of the GPU. It downloads nothing: the Sentinel-2 data must already be in Drive as
`MyDrive/garimpo/data.tar.zst`.

## How to use it

1. Open the notebook as "Open the notebook" says, on a GPU runtime: Runtime > Change runtime type.
   Changing the runtime later starts a new session, which erases the disk.
2. Run Runtime > Run all and keep the tab open. The last cell writes the pending files to Drive
   and releases the GPU.
3. If the session ends before the last cell, open a new session and run Runtime > Run all again.
   Finished stages are skipped, and a training continues from its last checkpoint.

The title of a cell that is not part of every run says so:

- "- (optional)": it runs unless you clear its `RUN` box.
- "- (in a reexecution)": it does nothing unless you check its `RUN` box. Use it after a code fix,
  to redo a run from a given stage.

| Section | What it does | Saved in Drive |
|---|---|---|
| 1. Setup | Mounts Drive, clones the code, builds the Python environment | Nothing |
| 2. Data | Unpacks the data and restores the earlier runs and logs | Nothing |
| 3. Run configs | Shows the values each run reads | Nothing |
| 4. Tests | Runs the test suite on the GPU | Nothing |
| 5. Pilot | Runs the whole pipeline on 1% of the training tiles, to check the code | `runs/pilot/`, `logs/` |
| 6. Training | Runs the four models on all the training tiles | `runs/<model>-f1.0/`, `logs/` |
| 7. Evaluation | Merges the four runs' result tables | `outputs/` |
| 8. Release | Writes the pending files to Drive and releases the GPU | Everything above |

## Open the notebook

Open the notebook from its GitHub link,
<https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/colab.ipynb>,
in every session, so the cells are always those on `main`. A copy made with File > Upload notebook
or File > Save a copy in Drive keeps the cells of the day it was made, and the clone does not update
them. Options changed in a form are not saved either: they stay set in the open tab, and reopening
the notebook from the link restores the values on `main`, where `FORCE_FROM` and `FORCE_VALIDATE`
are empty.

The session clones the repository without credentials and cannot push, so change the code on your
own machine and push it from there. The next session clones the new version.

## 1. Setup

The runtime's disk is erased when a session ends. These cells clone the code again and build the
Python environment; section 2 copies the data and the earlier runs back from Drive.

Mount Google Drive. Colab asks for permission the first time. Everything that must outlive the
session is kept in `MyDrive/garimpo`.

In [ ]:
# @title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

The paths and variables the other cells use. `DATA` must stay `/content/garimpo-data`: every run
records the path of its data and refuses to continue when it is restored under another path.

In [ ]:
# @title Settings
import os
import subprocess
import threading
import time
from pathlib import Path

from google.colab import runtime

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions
DATA = "/content/garimpo-data"  # runs record this path, so it never changes

os.environ["GARIMPO_DATA_DIR"] = DATA
periodic_copy = None  # the copy to Drive every 30 minutes of step 3, once started

The functions the steps use. Every pilot and full-run job ends by printing `PIPELINE CHECK: PASS`
or `PIPELINE CHECK: FAIL`, and a cell keeps going when one of its shell commands fails, so read
that line to know how a job went. A job's log keeps every attempt: `start_log` marks where a new
one begins, and `print_verdicts` prints the verdict of the last attempt only.
`copy_to_drive_every_30_min` saves the runs and logs during the pilot and the training.
`release_runtime` ends the notebook: it writes the files still pending to Drive, then releases the
machine. When a Python error stops a cell, Colab cancels the cells queued after it, release
included. No cell before the release raises one: a failed shell command only prints its error, and
the Python cells that can fail print theirs. If you interrupt a cell yourself, run the cells of
section 8 by hand.

In [ ]:
# @title Define the helper functions
VERDICTS = ("PIPELINE CHECK: PASS", "PIPELINE CHECK: FAIL")
ATTEMPT = "=== attempt started "


def start_log(name: str) -> None:
    """Mark the start of a new attempt in the job's log. It never raises."""
    started = time.strftime("%Y-%m-%d %H:%M", time.gmtime())
    try:
        with (Path("outputs/logs") / f"{name}.log").open("a", encoding="utf-8") as log:
            log.write(f"{ATTEMPT}{started} UTC ===\n")
    except OSError as error:
        print(f"Could not mark the attempt in the log: {error}")


def print_verdicts(log_names: list[str]) -> None:
    """Print the pipeline-check line of the last attempt in each log. It never raises."""
    for name in log_names:
        log = Path("outputs/logs") / f"{name}.log"
        try:
            text = log.read_text(encoding="utf-8", errors="replace")
        except OSError as error:
            print(name, f"NO LOG: {error}")
            continue
        text = text.rsplit(ATTEMPT, 1)[-1]
        found = [line for line in text.splitlines() if line.startswith(VERDICTS)]
        print(name, found[-1] if found else "NO VERDICT: the job or its check died; see the log")


def copy_to_drive_every_30_min(stop: threading.Event) -> None:
    """Copy runs/ and the logs to Drive every 30 minutes until stop is set."""
    runs = ["rsync", "-a", "--exclude", "*.tmp", "runs/", f"{DRIVE}/runs/"]
    logs = ["rsync", "-a", "outputs/logs/", f"{DRIVE}/logs/"]
    while not stop.wait(1800):
        for command in (runs, logs):
            try:
                copied = subprocess.run(command, check=False)  # noqa: S603
            except OSError as error:
                print(f"The copy to Drive every 30 minutes failed: {error}")
                continue
            if copied.returncode != 0:
                print(f"The copy to Drive every 30 minutes ended with code {copied.returncode}")


def release_runtime() -> None:
    """Write pending files to Drive, then release the runtime, which ends the session."""
    try:
        drive.flush_and_unmount()
    except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
        print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()

`rsync` copies the runs to and from Drive, and `zstd` unpacks the data archive. The
pipeline compares each input file's size and modification time to tell which stages are already
done, and both `rsync -a` and `tar` keep those times.

In [ ]:
# @title Install rsync and zstd
!apt-get -qq install -y rsync zstd > /dev/null

`uv` builds the Python environment from `uv.lock`, which pins every package version.

In [ ]:
# @title Install uv
!pip install -q uv

Clone the code with its `compass/` submodule, the adapted COMPASS method. A session starts in
`/content`, so git clones into `/content/garimpo-uncertainty`, the `CODE` folder the next cell
enters. Run it once per session: run again after that cell, it clones a second copy inside `CODE`.

In [ ]:
# @title Clone the code
!git clone --recurse-submodules {REPO}

In [ ]:
# @title Enter the code folder
%cd {CODE}

In [ ]:
# @title Build the Python environment
!uv sync

The commit this session runs. Note it next to any result you keep.

In [ ]:
# @title Show the commit
!git log -1 --format='%h %cs %s'

In [ ]:
# @title Create the folders
!mkdir -p {DRIVE}/runs {DRIVE}/logs {DATA} runs outputs/logs

## 2. Data

Unpack the data archive, `MyDrive/garimpo/data.tar.zst`, which must already be in Drive. The
unpacked store always takes 69.2 GB, whatever the training fraction, because its arrays are created
for all 18,864 cells. `tar` lists each file as it unpacks it, and reports the archive missing if it
is not in Drive.

In [ ]:
# @title Unpack the data
!tar -I zstd -xvf {DRIVE}/data.tar.zst -C {DATA}

Copy back the runs and logs of earlier sessions. The finished stages of a run are then skipped,
and each job's log keeps the earlier attempts above the new one, including one cut short by a
session that ended. If one of these two cells fails,
fix it before running a step: a step that runs without its earlier runs starts them over, and its
copy to Drive replaces the saved ones.

In [ ]:
# @title Restore the runs
!rsync -a {DRIVE}/runs/ runs/

In [ ]:
# @title Restore the logs
!rsync -a {DRIVE}/logs/ outputs/logs/

Free disk. The store needs 69.2 GB and the Python environment 6.9 GB, plus room for the runs.

In [ ]:
# @title Show the free disk
!df -h /content

## 3. Run configs

Each run reads every value from its JSON file in `configs/`, versioned with the code:
`configs/pilot/` holds the three pilot runs and `configs/full/` the four models. The pilot keeps the
models' values except where it must stay short or reach the last stage: 1% of the training tiles,
fewer epochs and steps, every layer accepted, and a coarser search threshold. To change a value,
edit the file on your own machine, commit and push it; the next session clones it. A job refuses a
config with a value missing, empty or misspelled before it runs any stage.

In [ ]:
# @title Show the run configs
import json

import pandas as pd

configs = {
    path.stem: json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(Path("configs").glob("*/*.json"))
}
display(pd.DataFrame(configs))

## 4. Tests

Optional after the first session: clear the `RUN` box of the cell to skip them. The tests build a
small synthetic dataset of their own. GitHub's CI runs them on a CPU; here they run on CUDA. A few
tests skip on a GPU on purpose, such as those that compare a result with one computed on the CPU,
and every other test should pass. A failure does not stop the cells after it, so read the output
of this cell before you trust the pilot.

In [ ]:
# @title Run the tests on the GPU - (optional)
RUN = True  # @param {type:"boolean"}
if RUN:
    !GARIMPO_TEST_DEVICE=cuda uv run --no-sync pytest -q

## 5. Pilot

The pilot runs the whole pipeline on 1% of the training tiles, with validation, calibration and
test whole, to find bugs on the GPU before the four models train. Its numbers say nothing about the
methods. Each U-Net starting point has its own cell, reading its own config. Each one writes
`runs/pilot/<config>/` and the log `outputs/logs/pilot-<config>.log`.

Copy `runs/` and the logs to Drive every 30 minutes from here to the end of the training, so a
session that ends early keeps the finished stages, the last checkpoint and the log up to that
point. Running this cell again while the copy runs starts nothing new.

In [ ]:
# @title Start the copy to Drive every 30 minutes
if periodic_copy is None or not periodic_copy.is_alive():
    stop_copy = threading.Event()
    periodic_copy = threading.Thread(
        target=copy_to_drive_every_30_min, args=(stop_copy,), daemon=True
    )
    periodic_copy.start()

In [ ]:
# @title Pilot of the U-Net from scratch
start_log("pilot-unet-none")
!NUM_WORKERS={os.cpu_count()} bash slurm/pilot.sbatch configs/pilot/unet-none.json 2>&1 | tee -a outputs/logs/pilot-unet-none.log

In [ ]:
# @title Pilot of the U-Net from ImageNet weights
start_log("pilot-unet-imagenet")
!NUM_WORKERS={os.cpu_count()} bash slurm/pilot.sbatch configs/pilot/unet-imagenet.json 2>&1 | tee -a outputs/logs/pilot-unet-imagenet.log

In [ ]:
# @title Pilot of the U-Net from Sentinel-2 MoCo weights
start_log("pilot-unet-sentinel2_moco")
!NUM_WORKERS={os.cpu_count()} bash slurm/pilot.sbatch configs/pilot/unet-sentinel2_moco.json 2>&1 | tee -a outputs/logs/pilot-unet-sentinel2_moco.log

After a code fix, push it, start a new session, and check the `RUN` box of the pilot to redo, with
`FORCE_FROM` set to the first stage the fix touches: that stage and the ones after it run again,
and the earlier ones are skipped. `FORCE_FROM=train` refuses while `last.pt` holds a training that
has not finished; run the pilot cell above instead, which resumes it. `FORCE_VALIDATE` checks the
data again.

In [ ]:
# @title Rerun the pilot of the U-Net from scratch - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = "train"  # @param ["train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log("pilot-unet-none")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/pilot.sbatch configs/pilot/unet-none.json 2>&1 | tee -a outputs/logs/pilot-unet-none.log

In [ ]:
# @title Rerun the pilot of the U-Net from ImageNet weights - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = "train"  # @param ["train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log("pilot-unet-imagenet")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/pilot.sbatch configs/pilot/unet-imagenet.json 2>&1 | tee -a outputs/logs/pilot-unet-imagenet.log

In [ ]:
# @title Rerun the pilot of the U-Net from Sentinel-2 MoCo weights - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = "train"  # @param ["train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log("pilot-unet-sentinel2_moco")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/pilot.sbatch configs/pilot/unet-sentinel2_moco.json 2>&1 | tee -a outputs/logs/pilot-unet-sentinel2_moco.log

One line per variant. `PASS` means every stage finished and every check held. After `FAIL`, the
lines above it in the log name the stage that failed and its error. `NO VERDICT` means the job or
its check died first, and the log shows where. `NO LOG` means that variant has no log yet, or
its log could not be read.

In [ ]:
# @title Print the verdicts of the pilot
print_verdicts(["pilot-unet-none", "pilot-unet-imagenet", "pilot-unet-sentinel2_moco"])

## 6. Training

The four models all use a ResNet-18 encoder and train on 100% of the training tiles, each from its
own config in `configs/full/`. Each run chooses the COMPASS-J layer on the validation set and
writes the reason to `chosen_layer.json`. A long training may not fit in one session: run the
notebook again in a new one, and each run resumes where it stopped.

In [ ]:
# @title Train the U-Net from scratch
start_log("unet_scratch-f1.0")
!NUM_WORKERS={os.cpu_count()} bash slurm/full.sbatch configs/full/unet_scratch-f1.0.json 2>&1 | tee -a outputs/logs/unet_scratch-f1.0.log

In [ ]:
# @title Train the U-Net from ImageNet weights
start_log("unet_imagenet-f1.0")
!NUM_WORKERS={os.cpu_count()} bash slurm/full.sbatch configs/full/unet_imagenet-f1.0.json 2>&1 | tee -a outputs/logs/unet_imagenet-f1.0.log

In [ ]:
# @title Train the U-Net from Sentinel-2 MoCo weights
start_log("unet_sentinel2-f1.0")
!NUM_WORKERS={os.cpu_count()} bash slurm/full.sbatch configs/full/unet_sentinel2-f1.0.json 2>&1 | tee -a outputs/logs/unet_sentinel2-f1.0.log

In [ ]:
# @title Train the ConvNet (DeepLabV3)
start_log("convnet-f1.0")
!NUM_WORKERS={os.cpu_count()} bash slurm/full.sbatch configs/full/convnet-f1.0.json 2>&1 | tee -a outputs/logs/convnet-f1.0.log

After a code fix, check the `RUN` box of the model to redo, with `FORCE_FROM` set to the first
stage the fix touches. The options work as in the pilot.

In [ ]:
# @title Rerun the U-Net from scratch - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = "train"  # @param ["train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log("unet_scratch-f1.0")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/full.sbatch configs/full/unet_scratch-f1.0.json 2>&1 | tee -a outputs/logs/unet_scratch-f1.0.log

In [ ]:
# @title Rerun the U-Net from ImageNet weights - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = "train"  # @param ["train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log("unet_imagenet-f1.0")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/full.sbatch configs/full/unet_imagenet-f1.0.json 2>&1 | tee -a outputs/logs/unet_imagenet-f1.0.log

In [ ]:
# @title Rerun the U-Net from Sentinel-2 MoCo weights - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = "train"  # @param ["train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log("unet_sentinel2-f1.0")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/full.sbatch configs/full/unet_sentinel2-f1.0.json 2>&1 | tee -a outputs/logs/unet_sentinel2-f1.0.log

In [ ]:
# @title Rerun the ConvNet (DeepLabV3) - (in a reexecution)
RUN = False  # @param {type:"boolean"}
FORCE_FROM = "train"  # @param ["train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
force_validate = "1" if FORCE_VALIDATE else ""
if RUN:
    start_log("convnet-f1.0")
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={force_validate} NUM_WORKERS={os.cpu_count()} \
        bash slurm/full.sbatch configs/full/convnet-f1.0.json 2>&1 | tee -a outputs/logs/convnet-f1.0.log

One line per model, as for the pilot.

In [ ]:
# @title Print the verdicts of the training
print_verdicts(["unet_scratch-f1.0", "unet_imagenet-f1.0", "unet_sentinel2-f1.0", "convnet-f1.0"])

## 7. Evaluation

The merge writes `results.csv` and the comparison tables to `outputs/`, from the four runs. It
needs all four to have passed. Download `MyDrive/garimpo/outputs/` afterwards and commit it from
your own machine: these are the final result tables.

In [ ]:
# @title Merge the four runs
!uv run --no-sync python scripts/merge_results.py --out outputs --runs \
    runs/unet_scratch-f1.0 runs/unet_imagenet-f1.0 runs/unet_sentinel2-f1.0 runs/convnet-f1.0

In [ ]:
# @title Save the tables in Drive
!rsync -a outputs/ {DRIVE}/outputs/

## 8. Release

Stop the copy every 30 minutes, then copy the runs and logs to Drive one last time and release the
GPU. The stop cell waits for a copy in progress to finish, so that it and the final copy never write
to Drive at the same time.

In [ ]:
# @title Stop the copy every 30 minutes
if periodic_copy is not None:
    stop_copy.set()
    periodic_copy.join()

In [ ]:
# @title Save the runs in Drive
!rsync -a runs/ {DRIVE}/runs/

In [ ]:
# @title Save the logs in Drive
!rsync -a outputs/logs/ {DRIVE}/logs/

In [ ]:
# @title Write pending files to Drive and release the GPU
release_runtime()

## Where the files are

| Path | Contents | Kept after the session |
|---|---|---|
| `MyDrive/garimpo/data.tar.zst` | The data, which the notebook only reads | Yes |
| `MyDrive/garimpo/runs/` | Every run: checkpoints, tables, stage statuses | Yes |
| `MyDrive/garimpo/logs/` | The log of every pilot and full-run job | Yes |
| `MyDrive/garimpo/outputs/` | The merged tables of section 7 | Yes |
| `/content/garimpo-uncertainty/` | The code | No |
| `/content/garimpo-data/` | The unpacked data | No |